# 02 · Prompt engineering  (Phase 2)

**Who Wants to Be a PoliMillionaire?** — three prompt strategies, head to head this notebook runs.

The baseline (notebook 01) scored **87%**, with the misses concentrated in **Maths** (arithmetic
errors) and one **Science** knowledge-cutoff trap. The central question here:

> Does **chain-of-thought** (think step by step) rescue the arithmetic on its own — or is a
> deterministic **calculator tool** (Phase 3) actually required?

Same engine, same dev set, same pipeline seam — **only the prompt strategy changes**:
`zero_shot_v1` · `few_shot_v1` · `cot_v1`. The model, loaded ONCE it is; three benchmarks then run.

> **GPU needed.** Runtime ▸ Change runtime type ▸ T4 GPU, select you must.

## 1 · Setup — clone the repo + install deps

In [ ]:
import os, sys

# The repo, into Colab we clone -- present already? Then update it instead, we do.
REPO_URL = 'https://github.com/SleepyEveryD/NLP.git'
REPO_ROOT = '/content/NLP'
if not os.path.exists(REPO_ROOT):
    !git clone {REPO_URL} {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull -q

SRC = os.path.join(REPO_ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)

In [ ]:
# The Phase-1/2 inference stack, install we do.
!pip install -q 'transformers>=4.45.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.43.0' sentencepiece einops pyyaml pandas matplotlib
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING -- no GPU. Runtime ▸ Change runtime type ▸ T4 GPU, switch you must.')

## 2 · Load the model once + the dev set
The 4-bit load (~1–2 min), paid a single time it is -- across all three strategies, reused the engine stays.

In [ ]:
import time
from config import RunConfig
from inference.engine import TransformersEngine
from classify.classifier import QuestionClassifier
from evaluation.dataset import load_questions

config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'base.yaml'))

# Once, the model we load.
t0 = time.perf_counter()
engine = TransformersEngine(
    model_name=config.model.name,
    quantization=config.model.quantization,
    dtype=config.model.dtype,
)
engine.warmup()
print(f'Model ready in {time.perf_counter() - t0:.1f}s')

# The collaborators that DO NOT change across strategies, here once we build.
classifier = QuestionClassifier()
questions = load_questions(os.path.join(REPO_ROOT, 'data', 'dev_questions.jsonl'))
print(f'Loaded {len(questions)} dev questions.')

## 3 · Run all three strategies
Per strategy, a fresh `run_id` and `PromptBuilder` we make -- everything else, identical it stays.
One logged run per strategy results.

In [ ]:
from dataclasses import replace
from prompting.builder import PromptBuilder
from agent.pipeline import QAPipeline
from evaluation.runner import BenchmarkRunner

strategies = ['zero_shot_v1', 'few_shot_v1', 'cot_v1']
run_paths = []
for s in strategies:
    # A fresh config per strategy -- a distinct run_id, its own log dir gives it.
    cfg = replace(config, run_id='phase2_' + s, prompt_strategy=s)
    pipe = QAPipeline(
        engine=engine,
        prompt_builder=PromptBuilder(strategy=s),
        classifier=classifier,
        latency_budget_s=config.latency_budget_s,
    )
    runner = BenchmarkRunner(pipe, cfg, log_root=os.path.join(REPO_ROOT, 'experiments', 'runs'))
    print('=== strategy:', s, '===')
    run_paths.append(runner.run(questions))
print()
print('Runs:', run_paths)

## 4 · Compare — overall, per-topic, and Maths in focus
The CoT-vs-tool question, the Maths column answers it.

In [ ]:
import pandas as pd
from evaluation.metrics import load_runs, accuracy_by

# All three runs, into one DataFrame we read.
df = load_runs(run_paths)

print('--- Overall accuracy by strategy ---')
print(accuracy_by(df, 'prompt_strategy').to_string(index=False))
print()

# Maths only -- does any strategy fix the arithmetic, see we do.
maths = df[df['topic'] == 'Maths']
print('--- MATHS accuracy by strategy ---')
print(accuracy_by(maths, 'prompt_strategy').to_string(index=False))
print()

# Topic x strategy, the full picture it paints.
known = df[df['correct'].notna()].copy()
known['correct'] = known['correct'].astype(float)
pivot = known.pivot_table(index='topic', columns='prompt_strategy', values='correct', aggfunc='mean')
print('--- Accuracy: topic x strategy ---')
print(pivot.round(2).to_string())
print()

# Latency + output length -- CoT costs tokens, confirm we do.
print('--- Mean latency (s) and mean tokens_out by strategy ---')
print(df.groupby('prompt_strategy')[['latency_s', 'tokens_out']].mean().round(2).to_string())

In [ ]:
import matplotlib.pyplot as plt

overall = accuracy_by(df, 'prompt_strategy').set_index('prompt_strategy')['accuracy']
maths_acc = accuracy_by(maths, 'prompt_strategy').set_index('prompt_strategy')['accuracy']
lat = df.groupby('prompt_strategy')['latency_s'].mean()
order = ['zero_shot_v1', 'few_shot_v1', 'cot_v1']

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(order, [overall.get(s, 0) for s in order], color='steelblue')
ax[0].set_ylim(0, 1); ax[0].set_title('Overall accuracy'); ax[0].set_ylabel('accuracy')
ax[1].bar(order, [maths_acc.get(s, 0) for s in order], color='seagreen')
ax[1].set_ylim(0, 1); ax[1].set_title('MATHS accuracy (the key question)')
ax[2].bar(order, [lat.get(s, 0) for s in order], color='darkorange')
ax[2].axhline(30.0, color='red', linestyle='--', label='30s budget')
ax[2].set_title('Mean latency (s)'); ax[2].legend()
for a in ax:
    a.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## 5 · Read-off + next step

_Fill in after the run:_

- **Best overall strategy:** ____  (and by how much over zero-shot?)
- **Did CoT fix Maths?** zero-shot Maths was 0.50. CoT Maths = ____ .
  - If CoT ≈ 1.0 → prompting alone handles arithmetic; the calculator tool becomes *reliability insurance*, not a necessity.
  - If CoT still misses → strong case for the **Phase 3 calculator** (deterministic, no token cost).
- **Latency cost of CoT:** mean latency / tokens_out vs zero-shot — still far under 30s? (it should be)
- **Prompt sensitivity:** which topics moved, which were flat? (feeds the rubric's prompt-sensitivity question)

Then: **Phase 3 (calculator tool)** if arithmetic still leaks, or **Phase 4 (RAG)** for the
knowledge-cutoff misses (e.g. the Science 'most moons' trap). Log the winning strategy into `experiments.md`.